# 05a — Manual verification of Places data

This is the manual-review entry point. The pipeline keeps every Google Places + OSM record, but the analysis filters out anything not marked `confirmed` (or `unsure`, by default).

Workflow:

1. **Bootstrap** — run cells 1–3 below. They write `audit_logs/manual_verification.csv` with one row per distinct place across both sources, status = `pending`. Re-running this notebook is idempotent: existing annotations are preserved, only new places get appended.
2. **Edit** — open `audit_logs/manual_verification.csv` in Excel / Numbers / VSCode and fill in the four editable columns:
    - `status`  →  one of `confirmed`, `rejected`, `unsure`, `duplicate`, `closed` (default `pending`).
    - `reviewer`  →  your initials.
    - `review_date_utc`  →  ISO date (YYYY-MM-DD).
    - `duplicate_of`  →  another `verification_id` if this row is a duplicate.
    - `notes`  →  optional free text.
3. **Apply** — run cell 5 below to see the current verification summary. Stage 6 (exposure measurement) calls `apply_verification` automatically, which keeps only `confirmed` + `unsure` rows by default and writes a per-run audit JSON to `audit_logs/`.

How to decide:
- `confirmed`  →  the place is a commercial premises with sunbeds (your judgement: website, Facebook, Google reviews mentioning sunbeds, etc.)
- `rejected`  →  it isn't (e.g. supermarket returned because it's near a salon, beauty salon with no sunbed service, hairdresser).
- `unsure`  →  can't tell from a quick lookup. Keeps the place in the analysis but flags it for sensitivity.
- `duplicate` →  same physical premises as another row; put the canonical `verification_id` in `duplicate_of`.
- `closed`  →  was a salon, has now closed.

You can do this in batches — the file is committed to git after each session so progress is durable.

## 1. Pre-flight

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import geopandas as gpd
import pandas as pd

from schools_sunbeds import audit, config, verification

config.ensure_dirs()
audit.verify_manifest()

VERIFICATION_CSV = config.AUDIT_LOGS / "manual_verification.csv"
VERIFICATION_CSV.relative_to(config.REPO_ROOT)

## 2. Load both salon sources

In [ ]:
google_files = sorted(config.DATA_PROCESSED.glob("salons_google_*.gpkg"))
osm_files = sorted(config.DATA_PROCESSED.glob("salons_osm_*.gpkg"))
google_gdf = gpd.read_file(google_files[-1]) if google_files else None
osm_gdf = gpd.read_file(osm_files[-1]) if osm_files else None
print(f"Google: {len(google_gdf) if google_gdf is not None else 0} places")
print(f"OSM   : {len(osm_gdf) if osm_gdf is not None else 0} features")

## 3. Bootstrap or update the verification CSV

Idempotent: existing rows are preserved verbatim, new places (from a fresh Stage 3 run) are appended as `pending`.

In [ ]:
_, audit_v = verification.update_verification_csv(
    VERIFICATION_CSV, google_gdf=google_gdf, osm_gdf=osm_gdf
)
print(f"Verification CSV: {VERIFICATION_CSV.relative_to(config.REPO_ROOT)}")
for k, v in audit_v.items():
    print(f"  {k:>22s}: {v}")

## 4. Current verification status

In [ ]:
ver = verification.load_verification(VERIFICATION_CSV)
summary = verification.verification_summary(ver)
print("Rows by source × status:")
print(summary.to_string())
print()
n_pending = int((ver["status"] == "pending").sum())
n_total = len(ver)
print(f"Outstanding manual reviews: {n_pending} / {n_total} ({n_pending*100/n_total:.0f}%)")

## 5. Preview what would be analysed if we ran the pipeline now

This calls `apply_verification` with the default keep-set (`confirmed` + `unsure`) and shows what survives. No filtering is persisted to disk by this cell — Stage 6 does that during its actual run.

In [ ]:
if google_gdf is not None:
    kept_google, audit_g = verification.apply_verification(
        google_gdf,
        ver,
        source="google",
        source_id_col="place_id",
    )
    print(f"Google: {audit_g['n_kept']} of {audit_g['n_input']} would be analysed")
if osm_gdf is not None:
    kept_osm, audit_o = verification.apply_verification(
        osm_gdf,
        ver,
        source="osm",
        source_id_col="osm_id",
    )
    print(f"OSM   : {audit_o['n_kept']} of {audit_o['n_input']} would be analysed")

## 6. Convenience views for review

Browse pending rows by LA, sorted by name. Useful for batch review.

In [ ]:
pending = ver[ver["status"] == "pending"]
if len(pending):
    print("Pending counts per LAD:")
    print(pending["lad_code"].value_counts().sort_index().to_string())
else:
    print("All places reviewed.")

In [ ]:
# Show the next 25 pending rows for review
pending.head(25)[["verification_id", "source", "name", "address", "lad_code", "imd_quintile"]]

## Done

Edit `audit_logs/manual_verification.csv`, then re-run cells 4–5 to see the updated summary. Stage 6 reads the same file at exposure-measurement time.